# L300 · Advanced Agent Evaluation, Production Monitoring & Regression Gate

This is the top of the LLMOps spine. In L100 you scored a tiny agent with one judge. In L200 you ran a judge suite behind a gateway. Here you operate the flagship **Multi-domain Supervisor** the way you would in production:

- **Offline evaluation** of the supervisor over a curated cross-domain eval set, with a suite of MLflow LLM judges.
- **Production (online) monitoring** that scores live traffic with the *same* judges, so dev and prod quality are measured on one ruler.
- **Unity Catalog lineage** across the agent, its three Genie spaces, the warehouse, and the Lakebase audit tables.
- **A regression gate** that compares a candidate against the current production baseline and blocks promotion on a score drop.

Everything is parametrized through notebook widgets, so it runs unchanged on any lab workspace (for example Vocareum) without editing code.

> Note: the data is synthetic. Product names, accounts, and suppliers are invented for the workshop.


## 0. Setup

Install MLflow 3.1+ and restart Python so the new version loads.


In [ ]:
%pip install -q mlflow databricks-langchain databricks-sdk databricks-agents
%restart_python


### Widgets — the only values you set

Every later cell reads these, so nothing is hardcoded to one workspace. Leave `catalog` blank to use the workspace `current_catalog()`. Point `experiment` at the experiment the supervisor logs traces to.


In [ ]:
dbutils.widgets.text('catalog', '', 'Unity Catalog (blank = current_catalog())')
dbutils.widgets.text('llm_endpoint', 'databricks-claude-sonnet-4-5', 'Judge / agent LLM endpoint')
dbutils.widgets.text('experiment_path', '', 'MLflow experiment path (blank = notebook experiment)')
dbutils.widgets.text('finance_space_id', '', 'Finance Genie space id (optional)')
dbutils.widgets.text('scm_space_id', '', 'SCM Genie space id (optional)')
dbutils.widgets.text('commercial_space_id', '', 'Commercial Genie space id (optional)')

catalog = dbutils.widgets.get('catalog').strip() or spark.sql('SELECT current_catalog()').first()[0]
llm_endpoint = dbutils.widgets.get('llm_endpoint').strip()
experiment_path = dbutils.widgets.get('experiment_path').strip()
print('catalog =', catalog)
print('llm_endpoint =', llm_endpoint)


In [ ]:
import mlflow
if experiment_path:
    mlflow.set_experiment(experiment_path)
print('experiment:', mlflow.get_experiment_by_name(experiment_path) if experiment_path else 'notebook default')


## 1. The cross-domain evaluation set

A small, curated set of questions that exercises each domain and the cross-domain fusion the supervisor exists for. In a real deployment this grows from production traces you label. Each row carries the question and the domains a correct answer must consult — the routing is itself something we evaluate.


In [ ]:
eval_set = [
    {'question': 'Why did gross margin for Paints EMEA fall in Q2 2026?',
     'expected_domains': ['FINANCE']},
    {'question': 'Which transport lanes had the worst OTIF last month?',
     'expected_domains': ['SCM']},
    {'question': 'Which key accounts are most at risk of churning this quarter?',
     'expected_domains': ['COMMERCIAL']},
    {'question': 'Did the EMEA margin slip line up with any supply or customer-risk signals?',
     'expected_domains': ['FINANCE', 'SCM', 'COMMERCIAL']},
    {'question': 'What is the capital of France?',  # out of scope
     'expected_domains': []},
]
print(f'{len(eval_set)} eval questions')


## 2. Call the supervisor under evaluation

We invoke the deployed supervisor serving endpoint (or, for a dev loop, the local `agent_server`). The `predict_fn` is the single seam MLflow's `evaluate` drives, so swapping a candidate model for the baseline is a one-line change.

Set `SUPERVISOR_ENDPOINT` to your served model name. If you are iterating locally, the fallback path imports the in-process supervisor from `agent_server/` instead.


In [ ]:
import os
from databricks.sdk import WorkspaceClient

SUPERVISOR_ENDPOINT = os.environ.get('SUPERVISOR_ENDPOINT', '').strip()
_w = WorkspaceClient()

def predict_fn(question: str) -> dict:
    """Return {'answer', 'routed_domains'} from the supervisor for one question."""
    if SUPERVISOR_ENDPOINT:
        resp = _w.serving_endpoints.query(
            name=SUPERVISOR_ENDPOINT,
            dataframe_records=[{'role': 'user', 'content': question}],
        )
        # Responses-API shaped output; pull the assistant text.
        text = str(resp.predictions if hasattr(resp, 'predictions') else resp)
        return {'answer': text, 'routed_domains': []}
    # Local dev fallback: in-process supervisor (needs the agent_server package on path).
    import sys; sys.path.insert(0, '../L300-usecase/agent_server')
    from agent_server import supervisor
    r = supervisor.supervise(question, persist=False)
    return {'answer': r['answer'], 'routed_domains': [x['domain'] for x in r['routing']]}


## 3. Offline evaluation with the MLflow judge suite

We score each answer with built-in LLM judges (relevance, safety, fluency) plus a custom **routing-correctness** scorer that checks the supervisor consulted the right domains. The judges use the same `llm_endpoint` as the agent, so quality is measured on one model family.


In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety, Guidelines

# A custom guideline scorer for the supervisor's core contract: ground answers in retrieved
# data, connect domains for cross-domain questions, and decline out-of-scope questions.
supervisor_guidelines = Guidelines(
    name='supervisor_contract',
    guidelines=(
        'The answer must be grounded only in retrieved data and must not invent figures. '
        'For a cross-domain question it must connect the domains rather than list them. '
        'For an out-of-scope question it must decline rather than guess.'
    ),
)

scorers = [RelevanceToQuery(), Safety(), supervisor_guidelines]
print('scorers:', [type(s).__name__ for s in scorers])


In [ ]:
import mlflow
from mlflow.genai import evaluate

def eval_data():
    for row in eval_set:
        yield {'inputs': {'question': row['question']},
               'expectations': {'expected_domains': row['expected_domains']}}

with mlflow.start_run(run_name='supervisor_offline_eval') as run:
    results = evaluate(
        data=list(eval_data()),
        predict_fn=lambda question: predict_fn(question)['answer'],
        scorers=scorers,
    )
    print('run_id:', run.info.run_id)
results.metrics if hasattr(results, 'metrics') else results


## 4. Routing correctness — did the supervisor consult the right domains?

A wrong-domain answer can still read fluently, so routing is evaluated on its own. We compare the domains the supervisor consulted against the expected set for each question and report precision on routing.


In [ ]:
def routing_score(row):
    out = predict_fn(row['question'])
    got = set(out.get('routed_domains') or [])
    want = set(row['expected_domains'])
    if not want:
        return 1.0 if not got else 0.0  # out-of-scope: should consult nothing / decline
    return len(got & want) / max(len(got | want), 1)

scores = {r['question']: routing_score(r) for r in eval_set}
for q, s in scores.items():
    print(f'{s:.2f}  {q[:70]}')
routing_accuracy = sum(scores.values()) / len(scores)
print('\nrouting_accuracy =', round(routing_accuracy, 3))


## 5. Production (online) monitoring

The same judges that scored the offline set run continuously over live production traces. Register them as scheduled scorers on the supervisor's experiment so every (or a sampled) production trace gets the same quality, safety, and grounding scores — dev parity on one ruler. The dashboard then shows quality, cost, and latency over time.

> The cell below registers the monitor. It is idempotent; re-running updates the sampling/scorer set.


In [ ]:
from mlflow.genai.scorers import ScorerSamplingConfig

try:
    for scorer in [RelevanceToQuery(), Safety(), supervisor_guidelines]:
        scorer.register(name=f'prod_{type(scorer).__name__.lower()}')
        scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=0.2))
    print('Registered production monitors at 20% sampling.')
except Exception as e:
    print('Online monitoring registration needs MLflow 3.x monitoring + an experiment with prod traffic.')
    print('Detail:', str(e)[:300])


## 6. Unity Catalog lineage

The supervisor's governance story is that every read is a governed query through a Genie space on a UC-managed warehouse, every action is a row in the UC-governed Lakebase tables, and the agent itself is a registered UC model. The query below surfaces the lineage that ties the agent to the three domain schemas it reads. Lineage is captured automatically by Unity Catalog when the agent runs governed SQL; here we read it back.


In [ ]:
for schema in ['akzo_finance', 'akzo_scm', 'akzo_commercial']:
    print(f'\n=== {catalog}.{schema} tables (read by the {schema.split("_")[1].upper()} sub-agent) ===')
    try:
        display(spark.sql(f'SHOW TABLES IN {catalog}.{schema}'))
    except Exception as e:
        print('  (schema not present in this lab):', str(e)[:160])


In [ ]:
# Read recent supervisor turns from the audit table (Lakebase), if the audit lineage is set up.
# In the workshop the audit lives in Lakebase; this UC view is the optional federated mirror.
try:
    display(spark.sql(f'SELECT * FROM {catalog}.akzo.agent_sessions ORDER BY created_at DESC LIMIT 20'))
except Exception as e:
    print('agent_sessions not federated into UC (it lives in Lakebase):', str(e)[:160])


## 7. The regression gate

Before a candidate supervisor (new prompt, new model, new routing description) is promoted, it must not regress the production baseline. We compare the candidate's aggregate quality + routing score against the current baseline and **fail the cell** if it drops beyond a tolerance. Wire this cell into a Databricks Job / CI step so promotion is blocked automatically.


In [ ]:
# Baseline score for the current production supervisor. In CI you read this from the registered
# model's tags or a metrics table; here we use a conservative workshop default you can override.
BASELINE_SCORE = float(os.environ.get('SUPERVISOR_BASELINE_SCORE', '0.80'))
TOLERANCE = 0.03  # allow small noise; block a real drop

# Candidate aggregate = mean of routing accuracy and a quality proxy (relevance pass-rate).
# Replace `quality_proxy` with the judge pass-rate from section 3 in a real gate.
quality_proxy = routing_accuracy  # stand-in when offline judges are unavailable in the lab
candidate_score = round((routing_accuracy + quality_proxy) / 2, 3)
print('baseline =', BASELINE_SCORE, ' candidate =', candidate_score)

if candidate_score < BASELINE_SCORE - TOLERANCE:
    raise ValueError(
        f'REGRESSION GATE FAILED: candidate {candidate_score} < baseline {BASELINE_SCORE} '
        f'- TOLERANCE {TOLERANCE}. Promotion blocked.'
    )
print('REGRESSION GATE PASSED — candidate is safe to promote.')


## What you built

- An offline evaluation of the supervisor with an MLflow judge suite plus a custom routing scorer.
- Production monitoring that scores live traffic with the same judges (dev/prod parity).
- A read of the Unity Catalog lineage across the three domain schemas and the audit tables.
- A regression gate that blocks promotion on a score drop.

This closes the loop the whole workshop has been building: **build → evaluate → govern → deploy → improve**, now at the scale of a multi-domain production agent.
